In [ ]:
from pyspark.sql.functions import col, count, when, isnan, lit, sum as spark_sum, min, max, avg, stddev
import json

# ============================================================
# DATA QUALITY FRAMEWORK FOR SILVER LAYER
# ============================================================

silver_table = spark.table("nyc_taxi.silver.green_taxi")
total_records = silver_table.count()

print(f"\n{'='*70}")
print(f"🔍 DATA QUALITY REPORT - nyc_taxi.silver.green_taxi")
print(f"{'='*70}")
print(f"Total Records: {total_records:,}\n")

quality_results = {}

# ============================================================
# CHECK 1: COMPLETENESS - Critical Fields Should Not Be Null
# ============================================================
print("\n[CHECK 1] COMPLETENESS - Null Values in Critical Fields")
print("-" * 70)

critical_fields = [
    "trip_id", "vendor_id", "pickup_datetime", "dropoff_datetime",
    "PULocationID", "DOLocationID", "trip_distance", "total_amount"
]

null_checks = []
for field in critical_fields:
    null_count = silver_table.filter(col(field).isNull()).count()
    null_pct = (null_count / total_records) * 100 if total_records > 0 else 0
    status = "✅ PASS" if null_count == 0 else "❌ FAIL"
    print(f"{status} | {field:25s} | Nulls: {null_count:6,} ({null_pct:.2f}%)")
    null_checks.append({"field": field, "null_count": null_count, "null_pct": null_pct})

quality_results["completeness"] = null_checks

# ============================================================
# CHECK 2: UNIQUENESS - Primary Key Check
# ============================================================
print("\n[CHECK 2] UNIQUENESS - Primary Key Validation")
print("-" * 70)

distinct_trip_ids = silver_table.select("trip_id").distinct().count()
duplicates = total_records - distinct_trip_ids
duplication_rate = (duplicates / total_records) * 100 if total_records > 0 else 0

if duplicates == 0:
    print(f"✅ PASS | trip_id is unique ({distinct_trip_ids:,} unique values)")
else:
    print(f"❌ FAIL | Found {duplicates:,} duplicate trip_ids ({duplication_rate:.2f}% duplication rate)")

quality_results["uniqueness"] = {"duplicates": duplicates, "duplication_rate": duplication_rate}

# ============================================================
# CHECK 3: VALIDITY - Business Rules & Data Ranges
# ============================================================
print("\n[CHECK 3] VALIDITY - Business Rules Compliance")
print("-" * 70)

validity_checks = [
    ("Negative fares", silver_table.filter(col("fare_amount") < 0).count()),
    ("Negative total amounts", silver_table.filter(col("total_amount") < 0).count()),
    ("Zero trip distance", silver_table.filter(col("trip_distance") == 0).count()),
    ("Trip duration > 24 hours", silver_table.filter(col("trip_duration_minutes") > 1440).count()),
    ("Trip duration <= 0", silver_table.filter(col("trip_duration_minutes") <= 0).count()),
    ("Passenger count = 0", silver_table.filter(col("passenger_count") == 0).count()),
    ("Passenger count > 6", silver_table.filter(col("passenger_count") > 6).count()),
    ("Pickup after dropoff", silver_table.filter(col("pickup_datetime") >= col("dropoff_datetime")).count()),
]

validity_results = []
for rule_name, violation_count in validity_checks:
    violation_pct = (violation_count / total_records) * 100 if total_records > 0 else 0
    status = "✅ PASS" if violation_count == 0 else "⚠️ WARN"
    print(f"{status} | {rule_name:30s} | Violations: {violation_count:6,} ({violation_pct:.2f}%)")
    validity_results.append({"rule": rule_name, "violations": violation_count, "violation_pct": violation_pct})

quality_results["validity"] = validity_results

# ============================================================
# CHECK 4: CONSISTENCY - Referential Integrity
# ============================================================
print("\n[CHECK 4] CONSISTENCY - Location ID Integrity")
print("-" * 70)

# Assuming valid NYC location IDs are between 1 and 265 (TLC standard)
invalid_pu = silver_table.filter((col("PULocationID") < 1) | (col("PULocationID") > 265)).count()
invalid_do = silver_table.filter((col("DOLocationID") < 1) | (col("DOLocationID") > 265)).count()

pu_status = "✅ PASS" if invalid_pu == 0 else "❌ FAIL"
do_status = "✅ PASS" if invalid_do == 0 else "❌ FAIL"

print(f"{pu_status} | Invalid PULocationID (outside 1-265) | Count: {invalid_pu:,}")
print(f"{do_status} | Invalid DOLocationID (outside 1-265) | Count: {invalid_do:,}")

quality_results["consistency"] = {
    "invalid_pickup_locations": invalid_pu,
    "invalid_dropoff_locations": invalid_do
}

# ============================================================
# CHECK 5: ACCURACY - Statistical Outliers
# ============================================================
print("\n[CHECK 5] ACCURACY - Statistical Summary")
print("-" * 70)

stats = silver_table.select(
    avg("fare_amount").alias("avg_fare"),
    stddev("fare_amount").alias("stddev_fare"),
    min("fare_amount").alias("min_fare"),
    max("fare_amount").alias("max_fare"),
    avg("trip_distance").alias("avg_distance"),
    max("trip_distance").alias("max_distance"),
    avg("trip_duration_minutes").alias("avg_duration"),
    max("trip_duration_minutes").alias("max_duration")
).collect()[0]

# Handle None values in stats
avg_fare = stats['avg_fare'] if stats['avg_fare'] is not None else 0.0
stddev_fare = stats['stddev_fare'] if stats['stddev_fare'] is not None else 0.0
min_fare = stats['min_fare'] if stats['min_fare'] is not None else 0.0
max_fare = stats['max_fare'] if stats['max_fare'] is not None else 0.0
avg_distance = stats['avg_distance'] if stats['avg_distance'] is not None else 0.0
max_distance = stats['max_distance'] if stats['max_distance'] is not None else 0.0
avg_duration = stats['avg_duration'] if stats['avg_duration'] is not None else 0.0
max_duration = stats['max_duration'] if stats['max_duration'] is not None else 0.0

print(f"Fare Amount:     Avg=${avg_fare:.2f} | StdDev=${stddev_fare:.2f} | Min=${min_fare:.2f} | Max=${max_fare:.2f}")
print(f"Trip Distance:   Avg={avg_distance:.2f} mi | Max={max_distance:.2f} mi")
print(f"Trip Duration:   Avg={avg_duration:.2f} min | Max={max_duration:.2f} min")

# Detect extreme outliers (3 standard deviations)
fare_upper_bound = avg_fare + (3 * stddev_fare) if stddev_fare > 0 else avg_fare + 100
extreme_fares = silver_table.filter(col("fare_amount") > fare_upper_bound).count()
extreme_distances = silver_table.filter(col("trip_distance") > 100).count()

print(f"\n⚠️ Extreme outliers detected:")
print(f"   - Fares > ${fare_upper_bound:.2f} (3σ): {extreme_fares:,} trips")
print(f"   - Distances > 100 miles: {extreme_distances:,} trips")

quality_results["accuracy"] = {
    "extreme_fares": extreme_fares,
    "extreme_distances": extreme_distances,
    "stats": {k: float(v) if v is not None else None for k, v in stats.asDict().items()}
}

# ============================================================
# SUMMARY
# ============================================================
print(f"\n{'='*70}")
print("📊 QUALITY SCORE SUMMARY")
print(f"{'='*70}")

total_checks = len(critical_fields) + 1 + len(validity_checks) + 2  # completeness + uniqueness + validity + consistency
passed_checks = (
    sum(1 for check in null_checks if check['null_count'] == 0) +
    (1 if duplicates == 0 else 0) +
    sum(1 for check in validity_results if check['violations'] == 0) +
    (1 if invalid_pu == 0 else 0) +
    (1 if invalid_do == 0 else 0)
)

quality_score = (passed_checks / total_checks) * 100
print(f"\nOverall Quality Score: {quality_score:.1f}% ({passed_checks}/{total_checks} checks passed)")

if quality_score >= 95:
    print("✅ EXCELLENT - Data is production-ready")
elif quality_score >= 80:
    print("⚠️ GOOD - Minor issues detected, review recommended")
else:
    print("❌ CRITICAL - Significant data quality issues require attention")

quality_results["summary"] = {
    "total_checks": total_checks,
    "passed_checks": passed_checks,
    "quality_score": quality_score
}

print(f"\n{'='*70}\n")